# 01 — Data Exploration

Explore Jacq's writing corpus to understand style, themes, and data quality before training.

In [ ]:
import json
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
from nltk.tokenize import sent_tokenize, word_tokenize
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

In [ ]:
# Load processed text files
processed_dir = Path('../data/processed')
texts = {}
for f in processed_dir.rglob('*.txt'):
    texts[f.relative_to(processed_dir)] = f.read_text(encoding='utf-8')

print(f'Loaded {len(texts)} files')
for name, text in list(texts.items())[:5]:
    print(f'  {name}: {len(text):,} chars, {len(text.split()):,} words')

In [ ]:
# Sentence length distribution
all_text = ' '.join(texts.values())
sentences = sent_tokenize(all_text)
sent_lengths = [len(word_tokenize(s)) for s in sentences]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(sent_lengths, bins=50, edgecolor='black', alpha=0.7)
ax.set_xlabel('Words per sentence')
ax.set_ylabel('Count')
ax.set_title(f"Sentence Length Distribution (n={len(sentences):,})")
ax.axvline(x=sum(sent_lengths)/len(sent_lengths), color='red', linestyle='--', label=f'Mean: {sum(sent_lengths)/len(sent_lengths):.1f}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Word frequency analysis
words = word_tokenize(all_text.lower())
words = [w for w in words if w.isalpha() and len(w) > 2]
word_freq = Counter(words)

print(f'Total words: {len(words):,}')
print(f'Unique words: {len(word_freq):,}')
print(f'Type-token ratio: {len(word_freq)/len(words):.4f}')
print(f'\nTop 30 words:')
for word, count in word_freq.most_common(30):
    print(f'  {word}: {count}')

In [ ]:
# Training data inspection
training_path = Path('../data/training/all.jsonl')
if training_path.exists():
    examples = [json.loads(line) for line in training_path.read_text().strip().split('\n')]
    print(f'Training examples: {len(examples)}')
    
    response_lengths = [len(ex['messages'][2]['content'].split()) for ex in examples]
    print(f'Response length: min={min(response_lengths)}, max={max(response_lengths)}, avg={sum(response_lengths)//len(response_lengths)}')
    
    # Show a random example
    import random
    ex = random.choice(examples)
    print(f'\n--- Random Example ---')
    print(f'User: {ex["messages"][1]["content"][:100]}')
    print(f'Assistant: {ex["messages"][2]["content"][:200]}...')
else:
    print('No training data yet. Run: python scripts/build_training_data.py')